# DeepExtractor Visualizations and Results

This notebook visualizes the plots and results that can be found in our paper

### Imports

Let's start by setting some plot styles to make things look nice.

In [7]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import scienceplots # Need to import science plots

plt.style.use(['science'])
plt.rcParams['text.usetex'] = False # Turn to true if you want nice Latex plots

Now for some more imports that will be used throughout the notebook

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [10]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import torch
from gwpy.timeseries import TimeSeries
from gwpy.frequencyseries import FrequencySeries
from pycbc.types import TimeSeries as TimeSeries_pycbc

### Parameters

In [11]:
SAMPLE_RATE = 4096
NYQUIST_FREQUENCY = SAMPLE_RATE / 2
DURATION = 2.
DATA_PATH = 'data/'
TIME_DOMAIN_DATA_PATH = os.path.join(DATA_PATH, 'pycbc_noise/time_domain/')
PATH_TO_GRAVITY_SPY_DATASETs = '/home/tom.dooney/GravitySpy_datasets/'

## An example of a tomte glitch

In [12]:
from deepextractor.utils.visualization import plot_q_transform

In [13]:
# Let's start by loading some Gravity Spy data. For the moment, you will need acess to CIT to run this. 
# We will consider an example from O3b
gspy_o3a_path = PATH_TO_GRAVITY_SPY_DATASETs+'data_o3a_high_confidence.csv'
gspy_o3b_path = PATH_TO_GRAVITY_SPY_DATASETs+'data_o3b_high_confidence.csv'

# data_o3a = pd.read_csv(gspy_o3a_path) # uncomment if you want to look at O3a glitches
data_o3b = pd.read_csv(gspy_o3b_path)
data_o3b_tomte = data_o3b[data_o3b.label=='Tomte']
data_o3b_tomte = data_o3b_tomte.drop_duplicates(subset='GPStime')
data_o3b_tomte.head(1)

FileNotFoundError: [Errno 2] No such file or directory: '/home/tom.dooney/GravitySpy_datasets/data_o3b_high_confidence.csv'

In [ ]:
# Let's grab the first tomte glitch we see in the dataset
j = 0
# We need the interferometer and the gps time of the glitch
ifo = data_o3b_tomte.ifo.iloc[j]
gps_time = data_o3b_tomte.GPStime.iloc[j]

# Fetch glitch data from the time of interest
# We grab 64s around the glitch to minimize the suppresion due to welch averaging (glitch will affect PSD)
glitch = TimeSeries.fetch_open_data(
    ifo,
    gps_time - 32,
    gps_time + 32,
    sample_rate=SAMPLE_RATE
)

# We whiten the data with PyCBC, so let's convert from Gwpy to PyCBC
# But PyCBC doesn't like Gwpy so let's first go to numpy
glitch = np.asarray(glitch)
glitch_pycbc = TimeSeries_pycbc(glitch, delta_t=1. / SAMPLE_RATE)
white_glitch, psd = glitch_pycbc.whiten(2, 1, remove_corrupted=False, return_psd=True)
white_glitch = np.asarray(white_glitch) # Let's go back to numpy as everyone likes him
white_glitch_centered = white_glitch[len(white_glitch) // 2 - 4096:len(white_glitch) // 2 + 4096] # Let's look at 2s around the glitch

# Now we can plot a time series and Gravity Spy Q-scan
t = np.arange(0,DURATION,1/SAMPLE_RATE)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# 1. Time series plot
ax[0].plot(t, white_glitch_centered)
ax[0].set_xlabel('Time [s]', fontsize=35)
ax[0].set_ylabel(r'$h(t)$', fontsize=35)
ax[0].grid(True)
ax[0].tick_params(axis='both', which='major', labelsize=25)

# 2. Q-scan
plot_q_transform(white_glitch_centered, srate=SAMPLE_RATE, whiten=False, ax=ax[1])
ax[1].set_xlabel('Time [s]', fontsize=35)
ax[1].set_ylabel('Frequency [Hz]', fontsize=35)
ax[1].tick_params(axis='both', which='major', labelsize=25)
plt.tight_layout()
plt.show()

## Simulated glitches for training DeepExtractor

Now that we have seen an example of a glitch, let's now plot some examples of our simulate, proxy glitches that we generate using analytical waveforms

In [ ]:
from simulated_glitch_functions import (
    generate_chirp, generate_sine, generate_sine_gaussian,
    generate_gaussian_pulse, ringdown, generate_gengli_glitch,
    generate_cdvgan_glitch
)

In [ ]:
duration = 2. # We consider 2s samples
max_freq = 64 # We keep a low max frequency for the purposes of visualization

# Here are our 5 training classes
signal_types = ['chirp', 'sine', 'sine_gaussian', 'gaussian_pulse', 'ringdown']
signal_function_map = {
    'chirp': generate_chirp(duration, f0_min=1, f0_max=5, f1_min=1, f1_max=max_freq),
    'sine': generate_sine(duration, freq_min=1, freq_max=32),
    'sine_gaussian': generate_sine_gaussian(duration, freq_max=max_freq),
    'gaussian_pulse': generate_gaussian_pulse(duration),
    'ringdown': ringdown(duration)
}

# Create plots for each class
fig, axes = plt.subplots(1, len(signal_types), figsize=(25, 5))
for i, signal_type in enumerate(signal_types):
    t, signal = signal_function_map[signal_type]
    axes[i].plot(t, signal)
    axes[i].set_title(signal_type, fontsize=20)
    axes[i].set_xlabel('Time (s)', fontsize=20)
    axes[i].set_ylabel('Amplitude', fontsize=20)
    axes[i].tick_params(axis='both', which='major', labelsize=18)
plt.tight_layout()
plt.show()

### Training data for DeepExtractor in time and time-frequency (STFT)

Let's now visualize the data used to train DeepExtractor. Starting with simulated background noise, representing n(t), we inject the above proxy waveforms in the time domain to yield h(t) = n(t) + g(t). We then use the STFT to generate complex spectrograms h(t, f) and n(t, f).

The complex spectrograms are subsequently decomponsed into their magnitude and phase components.

In [ ]:
# These are functions required to generate synthetic time-domain data for training. The generation of correspoding spectrograms will come after
from generate_training_data import generate_gaussian_noise, generate_synthetic_data 
from sklearn.preprocessing import StandardScaler # Scaler to prepare data for training DeepExtractor

In [ ]:
# Let's start by defining some parameters
SAMPLE_RATE = 4096
NYQUIST_FREQ = SAMPLE_RATE // 2
T_MIN, T_MAX = 0.125, 2  # Duration range for glitch signals
T = 2.0  # Total duration in seconds
T_INJ = T / 2  # Injection time midpoint
LENGTH = int(T * SAMPLE_RATE)
MEAN = 0
STD_DEV = np.sqrt(SAMPLE_RATE/2) # PyCBC standards
NUM_SAMPLES = 10 # Let's just generate a few examples
BILBY_NOISE = False # We will generate data using numpy (PyCBC standards) which is used for the DeepLearning comparison. We generate noise with Bilby for our BayesWave comparison, due to BayesWave's requirements. They are essentially the same up to a normalization factor; PyCBC likes variance = SAMPLE_RATE/2, Bilby likes (unit) variance = 1.
SNR_MIN, SNR_MAX = 1, 250  # SNR range for glitch scaling
MINIMUM_FREQUENCY = 20. # This is for BayesWave comparison
SNR_SCALING_FACTOR_BILBY = 31.970149253731343 # This scaling factor is used if we need to 

In [ ]:
# We will redefine the signal_function_map to reflect the entire parameter space. Spefically, we consider the entire frequency range up to the Nyquist frequency (default parameters of these functions)
signal_types = ['chirp', 'sine', 'sine_gaussian', 'gaussian_pulse', 'ringdown']
signal_function_map = {
    'chirp': generate_chirp,
    'sine': generate_sine,
    'sine_gaussian': generate_sine_gaussian,
    'gaussian_pulse': generate_gaussian_pulse,
    'ringdown': ringdown
}

# We start with our Gaussian noise samples using generate_gaussian_noise, then we make the injections (between 1 and 30 per background sample) with generate_synthetic_data
gaussian_noise_samples = generate_gaussian_noise(MEAN, STD_DEV, NUM_SAMPLES, (LENGTH,), BILBY_NOISE)
h_t, n_t = generate_synthetic_data(gaussian_noise_samples, DATA_PATH, BILBY_NOISE, phase='examples') # There is some redundancy here in generate_synthetic_data as it outputs n_t; n_t == gaussian_noise_samples. Will clean this up later.
# We plot h_t and n_t below

# Here is an example of how we standardize the data using StandardScaler. StandardScaler rescales the data according to mean of 0 and unit variance. Standardization is required to make the data to make it suitable for training DeepLearning models
# Specifically, we fit the scaler to the h_t = n_t + g_t input mixture and transform it. We then use this fitted scaler to transform the n_t component without fitting. This means that h_t and n_t are standardized using the same statistics, and thus, occupy the same range
scaler = StandardScaler()
h_t_scaled = scaler.fit_transform(h_t.reshape(-1, 1)).reshape(h_t.shape)
n_t_scaled = scaler.transform(n_t.reshape(-1, 1)).reshape(n_t.shape)

# When actually training the models, we also require validation data for h_t and n_t. We take this fitted scaler above and use it to rescale the validation data without fitting. 
# We don't include validation data when fitting the scaler to prevent information leakage from the validation set into the training set.

In [ ]:
# Now let's generate the time-frequency (spectrogram) data, h(t, f) and n(t, f), from h(t) and n(t).
# We do have scripts that do this already, but due to the large memory required for generating full training sets, there are some intermediate steps where data is created and deleted where required.
# As a result, we will just process one example below following these steps.

# Let's start by defining some STFT parameters. We will choose those to generate 129x129 specs
n_fft = 256  
win_length = n_fft // 2  
hop_length = win_length // 2  
window = torch.hann_window(win_length)

# Extract the first index of the time series samples
h_t_sample = h_t[0]
n_t_sample = n_t[0]

h_t_tensor = torch.tensor(h_t, dtype=torch.float32)
n_t_tensor = torch.tensor(n_t, dtype=torch.float32)

# STFT computation
h_stft_tensor = torch.stft(h_t_tensor, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window, return_complex=True)
n_stft_tensor = torch.stft(n_t_tensor, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window, return_complex=True)

# Compute magnitude and phase
h_stft_magnitude = torch.abs(h_stft_tensor).numpy()  
h_stft_phase = torch.angle(h_stft_tensor).numpy()  
n_stft_magnitude = torch.abs(n_stft_tensor).numpy()
n_stft_phase = torch.angle(n_stft_tensor).numpy()

sample_index = 0  # Choose the first sample as an example
time_td = np.linspace(0, len(h_t_sample) / SAMPLE_RATE, len(h_t_sample))

# Time and frequency axes
num_time_bins = h_stft_magnitude.shape[2]  # Number of time bins
time = np.linspace(0, 2, num_time_bins)  # Time from 0 to 2 seconds
num_freq_bins = h_stft_magnitude.shape[1]  # Number of frequency bins
freq = np.linspace(0, NYQUIST_FREQUENCY, num_freq_bins)  # Frequency from 0 to Nyquist frequency

# Calculate dB magnitude for glitch and background, and find shared color scale limits
h_stft_magnitude_sample = h_stft_magnitude[sample_index]
h_stft_phase_sample = h_stft_phase[sample_index]
n_stft_magnitude_sample = n_stft_magnitude[sample_index]
n_stft_phase_sample = n_stft_phase[sample_index]

# Convert magnitudes to dB
h_stft_magnitude_sample_db = 20 * np.log10(h_stft_magnitude_sample + 1e-8)
n_stft_magnitude_sample_db = 20 * np.log10(n_stft_magnitude_sample + 1e-8)

# Determine global color scale limits across both glitch and background samples
vmin, vmax = min(h_stft_magnitude_sample_db.min(), n_stft_magnitude_sample_db.min()), max(h_stft_magnitude_sample_db.max(), n_stft_magnitude_sample_db.max())

# Create a grid layout
fig = plt.figure(figsize=(16, 18))
gs = GridSpec(4, 4, figure=fig)  # Create a 4x4 grid

# Input time series spanning across the top
ax_input_ts = fig.add_subplot(gs[0, 1:3])
ax_input_ts.plot(time_td, h_t_sample)
ax_input_ts.set_title("Input (h(t)) Time Series", fontsize=24)
ax_input_ts.set_xlabel("Time [s]", fontsize=20)
ax_input_ts.set_ylabel("Amplitude", fontsize=20)
ax_input_ts.grid(True)

# Input Magnitude Spectrogram
ax_input_magnitude = fig.add_subplot(gs[1, 0:2])
im1 = ax_input_magnitude.imshow(h_stft_magnitude_sample_db, aspect='auto', origin='lower', cmap='magma',
                                extent=[time[0], time[-1], freq[0], freq[-1]], vmin=vmin, vmax=vmax)
ax_input_magnitude.set_title("Input (h(t)) Magnitude Spectrogram", fontsize=24)
ax_input_magnitude.set_ylabel('Frequency [Hz]', fontsize=20)
ax_input_magnitude.set_xlabel('Time [s]', fontsize=20)
cb1 = fig.colorbar(im1, ax=ax_input_magnitude)
cb1.set_label('Magnitude [dB]', fontsize=18)

# Input Phase Spectrogram
ax_input_phase = fig.add_subplot(gs[1, 2:])
im2 = ax_input_phase.imshow(h_stft_phase_sample, aspect='auto', origin='lower', cmap='twilight',
                            extent=[time[0], time[-1], freq[0], freq[-1]], vmin=-np.pi, vmax=np.pi)
ax_input_phase.set_title("Input (h(t)) Phase Spectrogram", fontsize=24)
ax_input_phase.set_ylabel('Frequency [Hz]', fontsize=20)
ax_input_phase.set_xlabel('Time [s]', fontsize=20)
cb2 = fig.colorbar(im2, ax=ax_input_phase)
cb2.set_label('Phase [radians]', fontsize=18)

# Target Magnitude Spectrogram
ax_target_magnitude = fig.add_subplot(gs[2, 0:2])
im3 = ax_target_magnitude.imshow(n_stft_magnitude_sample_db, aspect='auto', origin='lower', cmap='magma',
                                 extent=[time[0], time[-1], freq[0], freq[-1]], vmin=vmin, vmax=vmax)
ax_target_magnitude.set_title("Target (n(t)) Magnitude Spectrogram", fontsize=24)
ax_target_magnitude.set_ylabel('Frequency [Hz]', fontsize=20)
ax_target_magnitude.set_xlabel('Time [s]', fontsize=20)
cb3 = fig.colorbar(im3, ax=ax_target_magnitude)
cb3.set_label('Magnitude [dB]', fontsize=18)

# Target Phase Spectrogram
ax_target_phase = fig.add_subplot(gs[2, 2:])
im4 = ax_target_phase.imshow(n_stft_phase_sample, aspect='auto', origin='lower', cmap='twilight',
                             extent=[time[0], time[-1], freq[0], freq[-1]], vmin=-np.pi, vmax=np.pi)
ax_target_phase.set_title("Target (n(t)) Phase Spectrogram", fontsize=24)
ax_target_phase.set_ylabel('Frequency [Hz]', fontsize=20)
ax_target_phase.set_xlabel('Time [s]', fontsize=20)
cb4 = fig.colorbar(im4, ax=ax_target_phase)
cb4.set_label('Phase [radians]', fontsize=18)

# Target time series spanning across the bottom
ax_target_ts = fig.add_subplot(gs[3, 1:3])
ax_target_ts.plot(time_td, n_t_sample)
ax_target_ts.set_title("Target (n(t)) Time Series", fontsize=24)
ax_target_ts.set_xlabel("Time [s]", fontsize=20)
ax_target_ts.set_ylabel("Amplitude", fontsize=20)
ax_target_ts.grid(True)

# Adjust layout for clarity
plt.tight_layout()
plt.show()


### Now let's look at PSDs

The PSD of a GW interferometer essentially describes the underlying noise properties of the detector. The data above uses simulated white noise, which has a flat and 'well-behaved' PSD. 

However, the PSD of real detector noise is more complicated and challenging to work with, even after whitening it. While whitening reduces broadband noise, narrow-band spectral lines—caused by electrical and mechanical sources or resonances—persist. Let's take a look!

In [ ]:
# Parameters
detector = 'H1'  # Choose 'L1' or 'H1'
start_time = 1256655618 + 10000  # Example GPS start time in O3b run
psd_duration = 512  # Duration for PSD calculation (in seconds)
slide_step = 1  # Sliding window step size (in seconds)
frequency_range = (10, 1600)  # Frequency range for plotting
mean = 0
std_dev = 1  # Use unit variance to comply with GwPy whitening filter, which normalizes to mean 0 and unit variance

# Segment parameters
segment_start = start_time
segment_end = segment_start + psd_duration

# Fetch data segment
data_segment = TimeSeries.fetch_open_data(detector, segment_start, segment_end, cache=True)

# Calculate the Power Spectral Density (PSD)
psd = data_segment.psd(fftlength=8, overlap=0)

# Plot the PSD
plt.figure(figsize=(10, 6))
plt.plot(psd.frequencies.value, psd.value, label=f"{detector} PSD")
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Frequency [Hz]", fontsize=22)
plt.ylabel("Power Spectral Density [1/Hz]", fontsize=22)
plt.xlim(frequency_range)
plt.ylim(bottom=1e-48)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.title("Real LIGO Hanford PSD", fontsize=24)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

# Whiten the data segment
whitened_segment = data_segment.whiten(fftlength=8, overlap=4)

# Calculate the PSD of the whitened data
whitened_psd = whitened_segment.psd(fftlength=8, overlap=4)

# Generate white Gaussian noise with the same properties as the segment
segment_length = psd_duration * SAMPLE_RATE  # Total number of samples in the segment
simulated_noise = np.random.normal(mean, std_dev, int(segment_length))

# Convert simulated noise to a GWPy TimeSeries object for PSD calculation
noise_timeseries = TimeSeries(simulated_noise, dt=1/SAMPLE_RATE)

# Calculate the PSD of the white Gaussian noise
simulated_noise_psd = noise_timeseries.psd(fftlength=8, overlap=4)

# Plot real vs simulated noise PSD
plt.figure(figsize=(10, 6))
plt.plot(whitened_psd.frequencies.value, whitened_psd.value, label='Real Noise', alpha=0.9)
plt.plot(simulated_noise_psd.frequencies.value, simulated_noise_psd.value, label='Simulated Noise', alpha=0.6)
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Frequency [Hz]", fontsize=22)
plt.ylabel("Power Spectral Density [1/Hz]", fontsize=22)
plt.xlim(frequency_range)
plt.ylim(bottom=1e-5)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.title("PSD of Whitened Hanford Data vs Simulated White Noise ", fontsize=24)
plt.legend(fontsize=20)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()


# Performance Evaluation

We first evaluate DeepExtractors performance on simulated data by considering the mismatch of injected and reconstructed test data.
We consider multiple test classes. These include the 5 training classes, as well as 7 unseen test classes from glitch generators gengli and cDVGAN.
As a conditional model, cDVGAN can create hybrid samples that blend features across all three training classes, introducing even greater diversity.

In [ ]:
import tensorflow as tf
import gengli
from pycbc.filter.matchedfilter import match
from simulated_glitch_functions import (
    generate_chirp, generate_sine, generate_sine_gaussian,
    generate_gaussian_pulse, ringdown, generate_gengli_glitch,
    generate_cdvgan_glitch, generate_gaussian_noise, load_tf_model
)
from simulated_evaluation import (
    generate_glitch_data, generate_hybrid_glitch_data,
    apply_stft, apply_istft, prepare_data_for_stft, 
    calculate_match, evaluate_model, 
)
from models.models import UNET1D, DnCNN1D, Autoencoder1D, Autoencoder2D, ModifiedAutoencoder2D, UNET2D

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
CHECKPOINT_DIR = 'checkpoints/'
OUTPUT_PATH = 'evaluation/'
DATA_DIR = 'data/'
SAMPLE_RATE = 4096
NYQUIST_FREQ = SAMPLE_RATE // 2
T_MIN, T_MAX = 0.125, 2
T = 2.0
T_INJ = T / 2
LENGTH = int(T * SAMPLE_RATE)
MEAN = 0
STD_DEV = 50
NUM_SAMPLES_PER_CLASS = 5
SNR_MIN, SNR_MAX = 7.5, 100


In [ ]:
# Model configurations
model_dict = {
    'UNET1D': UNET1D(in_channels=1, out_channels=1),
    'UNET1D_glitch': UNET1D(in_channels=1, out_channels=1),
    'UNET1D_diff': UNET1D(in_channels=1, out_channels=2),
    'UNET1D_2channel': UNET1D(in_channels=1, out_channels=2),
    'DnCNN1D': DnCNN1D(),
    'Autoencoder1D': Autoencoder1D(in_channels=1, out_channels=1),
    'Autoencoder2D': Autoencoder2D(in_channels=2, out_channels=2),
    'ModifiedAutoencoder2D': ModifiedAutoencoder2D(in_channels=2, out_channels=2),
    'DeepExtractor_65': UNET2D(in_channels=2, out_channels=2),
    'DeepExtractor_129': UNET2D(in_channels=2, out_channels=2),
    'DeepExtractor_257': UNET2D(in_channels=2, out_channels=2)
}


# Load cDVGAN model
glitch_generator = 'cdvgan'
try:
    generator = load_tf_model(DATA_DIR, glitch_generator)
except Exception as e:
    logger.error(f"Error loading cDVGAN generator: {e}")
    raise
    
# Signal function map
signal_function_map = {
    'chirp': generate_chirp,
    'sine': generate_sine,
    'sine_gaussian': generate_sine_gaussian,
    'gaussian_pulse': generate_gaussian_pulse,
    'ringdown': ringdown,
    'gengli_H1': lambda: generate_gengli_glitch(ifo='H1'),
    'gengli_L1': lambda: generate_gengli_glitch(ifo='L1'),
    'cdvgan_blip': lambda: generate_cdvgan_glitch('blip', generator),
    'cdvgan_tomte': lambda: generate_cdvgan_glitch('tomte', generator),
    'cdvgan_bbh': lambda: generate_cdvgan_glitch('bbh', generator),
    'cdvgan_simplex': lambda: generate_cdvgan_glitch('simplex', generator),
    'cdvgan_uniform': lambda: generate_cdvgan_glitch('uniform', generator)
}

In [ ]:
glitch_data = {}

# Generate glitch data for regular signals
signal_types = list(signal_function_map.keys())
for signal_type in signal_types:

    # Generate glitch data for each signal type
    glitch_data[signal_type] = generate_glitch_data(signal_type, gaussian_noise_samples, signal_function_map)

glitch_data['hybrid'] = generate_hybrid_glitch_data(gaussian_noise_samples, signal_function_map)

# Main loop for model evaluation (this loop can now use both regular and hybrid glitch data)
results_dict = {}
results_dict['data'] = glitch_data
results_dict['model_outputs'] = {}

model_name = 'DeepExtractor_257'
model = UNET2D(in_channels=2, out_channels=2)

# You can evaluate with either regular or hybrid glitch data (or both)
model_data_dict = evaluate_model(model_name, model_dict[model_name], scaler, results_dict['data'], OUTPUT_PATH, CHECKPOINT_DIR)
results_dict['model_outputs'][model_name] = model_data_dict

In [ ]:
# Sample signal classes (from your dict keys)
signal_classes = ['chirp', 'sine', 'sine_gaussian', 'gaussian_pulse', 'ringdown', 
                  'gengli_H1', 'gengli_L1', 'cdvgan_blip', 'cdvgan_tomte', 
                  'cdvgan_bbh', 'cdvgan_simplex', 'cdvgan_uniform']

# Create a 4x3 grid of subplots
fig, axes = plt.subplots(4, 3, figsize=(20, 15))  # 4 rows, 3 columns

# Flatten axes for easier iteration
axes = axes.flatten()

# Assuming `data_dict` contains the data
for i, signal_class in enumerate(signal_classes):
    # Extract data for the current signal class (using the first example, index 0)
    noisy_glitch_ts = results_dict['data'][signal_class]['noisy_glitch_ts'][0]
    clean_glitch_subtract = results_dict['data'][signal_class]['clean_glitch_subtract'][0]
    extracted_glitches = results_dict['model_outputs'][model_name][signal_class]['time_series']['extracted_glitches'][0]
    match_result = results_dict['model_outputs'][model_name][signal_class]['metrics']['match_glitch'][0]
    mismatch_result = (1-match_result)*100

    # Generate time array
    t = np.linspace(0, 2, len(clean_glitch_subtract))

    # Select the appropriate axis
    ax = axes[i]

    # Plot signals
    ax.plot(t, noisy_glitch_ts, c='gray', label='Noise+glitch', alpha=0.4)
    ax.plot(t, extracted_glitches, label='Extracted glitch', c='C2', linewidth=2, alpha=0.8)
    ax.plot(t, clean_glitch_subtract, label='Injected glitch', c='k', linestyle=':', linewidth=2, alpha=0.7)

    # Set titles, labels, and formatting
    ax.set_title(rf"{signal_class}, $\mathcal{{M}} = {mismatch_result.round(2)}\%$", fontsize=16)
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Amplitude', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(True)

    # Add legend only to the first subplot for clarity
    if i == 3:
        ax.legend(loc='upper right', fontsize=15)

# Remove unused subplots (if any)
for j in range(len(signal_classes), len(axes)):
    fig.delaxes(axes[j])

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Assuming `results_dict` contains the data and 'hybrid' is a valid signal class
hybrid_examples = 4  # Number of hybrid examples to plot

# Create a single-row grid of 4 subplots
fig, axes = plt.subplots(1, hybrid_examples, figsize=(20, 5))  # 1 row, 4 columns

# Iterate over the first 4 hybrid examples
for i in range(hybrid_examples):
    # Extract data for the current hybrid example
    noisy_glitch_ts = results_dict['data']['hybrid']['noisy_glitch_ts'][i]
    clean_glitch_subtract = results_dict['data']['hybrid']['clean_glitch_subtract'][i]
    extracted_glitches = results_dict['model_outputs'][model_name]['hybrid']['time_series']['extracted_glitches'][i]
    match_result = results_dict['model_outputs'][model_name]['hybrid']['metrics']['match_glitch'][i]
    mismatch_result = (1 - match_result) * 100

    # Generate time array
    t = np.linspace(0, 2, len(clean_glitch_subtract))

    # Select the appropriate axis
    ax = axes[i]

    # Plot signals
    ax.plot(t, noisy_glitch_ts, c='gray', label='Noise+glitch', alpha=0.4)
    ax.plot(t, extracted_glitches, label='Extracted glitch', c='C2', linewidth=2, alpha=0.8)
    ax.plot(t, clean_glitch_subtract, label='Injected glitch', c='k', linestyle=':', linewidth=2, alpha=0.7)

    # Set titles, labels, and formatting
    ax.set_title(rf"Hybrid {i+1}, $\mathcal{{M}} = {mismatch_result.round(2)}\%$", fontsize=14)
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Amplitude', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(True)

    # Add legend only to the first subplot for clarity
    if i == 0:
        ax.legend(loc='upper right', fontsize=15)

# Adjust layout
plt.tight_layout()
plt.show()


### BayesWave Comparison

For this part, we need to load data from another directory. Access to CIT is required.

In [ ]:
from pycbc.filter import match
import seaborn as sns

In [ ]:
BW_RESULTS_PATH = '/home/tom.dooney/BayesWave_distillation/BayesWave_Validation_Runs/'

In [ ]:
with open(BW_RESULTS_PATH+'BW_comparison_data.pkl', 'rb') as f:
    BW_comparison_data = pickle.load(f)

BW_comparison_data.keys()

In [ ]:
BW_comparison_data['cdvgan_bbh'][0].keys()

In [ ]:
# Initialize results list
mismatch_results = []

for gtype, samples in BW_comparison_data.items():
    for idx, data in samples.items():
        try:
            # Convert to PyCBC TimeSeries
            groundtruth = TimeSeries_pycbc(data['Groundtruth_glitch'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            bayeswave = TimeSeries_pycbc(data['BayesWave_glitch'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            deepextractor = TimeSeries_pycbc(data['DeepExtractor_glitch'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            
            # Compute mismatches
            match_bw = match(groundtruth, bayeswave)[0]
            match_deep = match(groundtruth, deepextractor)[0]
            
            mismatch_results.append({
                "Glitch Type": gtype,
                "Index": idx,
                "BayesWave Mismatch": round((1 - match_bw)*100, 3),
                "DeepExtractor Mismatch": round((1 - match_deep)*100, 3)
            })
        except Exception as e:
            print(f"Error processing {gtype} index {idx}: {e}")

# Convert to DataFrame
mismatch_df = pd.DataFrame(mismatch_results)

In [ ]:
import pandas as pd

# Define the desired order
desired_order = [
    "chirp",
    "sine",
    "sine_gaussian",
    "pulse",
    "ringdown",
    "gengli_H1",
    "gengli_L1",
    "cdvgan_blip",
    "cdvgan_tomte",
    "cdvgan_bbh",
    "cdvgan_simplex",
    "cdvgan_uniform"
]

# Convert 'Glitch Type' to a categorical variable with the desired order
mismatch_df['Glitch Type'] = pd.Categorical(mismatch_df['Glitch Type'], categories=desired_order, ordered=True)

# Group by 'Glitch Type' and calculate statistics
mismatch_mean_df = mismatch_df.groupby('Glitch Type')[['BayesWave Mismatch', 'DeepExtractor Mismatch']].mean().reset_index()
mismatch_std_dev_df = mismatch_df.groupby('Glitch Type')[['BayesWave Mismatch', 'DeepExtractor Mismatch']].std().reset_index()
mismatch_median_df = mismatch_df.groupby('Glitch Type')[['BayesWave Mismatch', 'DeepExtractor Mismatch']].median().reset_index()

# Sort DataFrames to ensure the order is reflected in the output
mismatch_mean_df = mismatch_mean_df.sort_values('Glitch Type').reset_index(drop=True)
mismatch_std_dev_df = mismatch_std_dev_df.sort_values('Glitch Type').reset_index(drop=True)
mismatch_median_df = mismatch_median_df.sort_values('Glitch Type').reset_index(drop=True)

# Display the resulting DataFrames
display(mismatch_median_df.round(2))


In [ ]:
mismatch_df['Glitch Type'] = mismatch_df['Glitch Type'].astype('category')

# Define y-axis limits
min_y = 0
max_y = 100

# plots_path = 'BW_comparison_plots/'
# os.makedirs(plots_path, exist_ok=True)
# filename = plots_path+'BW_mismatch_comparison_box.pdf'

# Set the figure size
plt.figure(figsize=(11, 8))

# Melt the data to long format for plotting
melted_data = pd.melt(mismatch_df, id_vars=['Glitch Type'],
                      value_vars=['BayesWave Mismatch', 'DeepExtractor Mismatch'],
                      var_name='source', value_name='mismatch')

# Create a box plot for both sources (split by Glitch Type)
sns.boxplot(x='Glitch Type', y='mismatch', hue='source',
            data=melted_data, palette='Set2', showfliers=False,
            width=0.58, dodge=True)  # Reduce box width and disable dodge to "melt" boxes together

# Title and labels
plt.ylabel('Mismatch (\%)', fontsize=32)
plt.xlabel('Glitch Type', fontsize=32)

# Rotate x-tick labels for better readability
plt.yticks(fontsize=25)
plt.xticks(rotation=45, ha='right', fontsize=28)  # Rotate x-axis labels
ax = plt.gca()  # Get current axes
# Make ticks extend only outward at the bottom, and disable top ticks
ax.tick_params(axis='x', bottom=True, top=False, direction='out', length=10)

# Set log scale on y-axis
plt.yscale('log')

# Adjust y-axis limits
plt.ylim(min_y, max_y)

# Add vertical lines passing through each x-tick label
xticks = ax.get_xticks()  # Get x-tick positions
# Add vertical lines that protrude slightly below the x-axis and stop at y = 100
for xtick in xticks:
    ax.vlines(xtick, min_y * 0.5, 100, linestyle='--', color='gray', alpha=1)


# Add grid lines for readability
plt.grid(True, axis='both', alpha=0.8)

# Add a legend with proper title
# plt.legend(loc='lower right', fontsize=20)
# plt.legend(title='Source', labels=['BayesWave', 'DeepExtractor'], loc='lower right', fontsize=20)
# Get the handles and labels from the legend
handles, _ = ax.get_legend_handles_labels()

# Manually set the legend labels and colors
ax.legend(handles=handles, labels=['BayesWave', 'DeepExtractor'], loc='lower right', fontsize=25)


# Tight layout for better spacing
plt.tight_layout()
# plt.savefig(filename, dpi=250, bbox_inches='tight')
plt.show()


Let's plot one example per class

In [ ]:
mismatch_df.groupby('Glitch Type')[['BayesWave Mismatch', 'DeepExtractor Mismatch']].mean().reset_index()

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

SNR_SCALING_FACTOR = 31.970149253731343

# Define the desired glitch types order
desired_order = [
    "chirp",
    "sine",
    "sine_gaussian",
    "pulse",
    "ringdown",
    "gengli_H1",
    "gengli_L1",
    "cdvgan_blip",
    "cdvgan_tomte",
    "cdvgan_bbh",
    "cdvgan_simplex",
    "cdvgan_uniform"
]

# Create a 4x3 grid for subplots
fig, axes = plt.subplots(4, 3, figsize=(18, 16))  # Adjust figure size as needed
axes = axes.flatten()  # Flatten to easily iterate over

# Iterate over glitch types and plot in subplots
for i, gtype in enumerate(desired_order):
    ax = axes[i]  # Select the corresponding subplot
    
    if gtype in BW_comparison_data:
        # Select a random sample for the glitch type
        random_idx = random.choice(list(BW_comparison_data[gtype].keys()))
        data = BW_comparison_data[gtype][random_idx]
        
        try:
            # Convert data to PyCBC TimeSeries to calculate matches. 
            groundtruth_normalized = TimeSeries_pycbc(data['Groundtruth_glitch'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            bayeswave_normalized = TimeSeries_pycbc(data['BayesWave_glitch'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            deepextractor_normalized = TimeSeries_pycbc(data['DeepExtractor_glitch'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            noisy_glitch_normalized = TimeSeries_pycbc(data['Input_data'], delta_t=1.0/SAMPLE_RATE, dtype='double')
            snr = data['SNR']
            
            # Time vector
            t_small = np.linspace(0, 2, len(groundtruth_normalized))  # Ensure time matches data length
            
            # Compute mismatches
            mismatch_BW = round((1 - match(groundtruth_normalized, bayeswave_normalized)[0]) * 100, 1)
            mismatch_DE = round((1 - match(groundtruth_normalized, deepextractor_normalized)[0]) * 100, 1)
            
            # Plotting
            ax.plot(t_small, noisy_glitch_normalized, c='gray', alpha=0.25)
            ax.plot(t_small, deepextractor_normalized, label=rf"DE, $\mathcal{{M}}={mismatch_DE}\%$", linewidth=2, c='C2', alpha=0.9)
            ax.plot(t_small, bayeswave_normalized, label=rf"BW, $\mathcal{{M}}={mismatch_BW}\%$", linewidth=2, c='C1', alpha=0.6)
            ax.plot(t_small, groundtruth_normalized, label="Injected", c='k', linewidth=1.8, linestyle=':', alpha=0.6)
            
            # Title for each subplot
            ax.set_title(f"{gtype} (SNR={round(snr * SNR_SCALING_FACTOR, 1)}), $\\mathcal{{M}}_{{\\text{{DE}}}}={mismatch_DE}\\%$, $\\mathcal{{M}}_{{\\text{{BW}}}}={mismatch_BW}\\%$", fontsize=17)
            
            # Axis labels only on left and bottom plots
            if i % 3 == 0:
                ax.set_ylabel("Normalized Amplitude", fontsize=22)
            if i >= 9:
                ax.set_xlabel("Time [s]", fontsize=22)
            
            # Grid and ticks
            ax.grid(True)
            ax.tick_params(axis='both', labelsize=20)
        
        except Exception as e:
            print(f"Error processing glitch type '{gtype}' at index {random_idx}: {e}")

# Adjust layout
plt.tight_layout()

# Create custom legend handles
custom_legend = [
    Line2D([0], [0], color='gray', lw=2, alpha=0.15, label='Noise + Glitch'),
    Line2D([0], [0], color='k', linestyle=':', lw=2, label='Injected'),                  # Black dashed for Injected
    Line2D([0], [0], color='C2', lw=2, label='DeepExtractor'),                          # Orange for DeepExtractor
    Line2D([0], [0], color='C1', lw=2, label='BayesWave')                               # Green for BayesWave
]

# Add custom legend to the figure
fig.legend(
    handles=custom_legend,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=4,
    fontsize=20
)

# Show the full figure
plt.show()


# A comparison of before and after Transfer Learning on real data

In [ ]:
from tqdm import tqdm
from utils import custom_whiten
from simulated_evaluation import (
    generate_glitch_data, generate_hybrid_glitch_data,
    apply_stft, apply_istft, prepare_data_for_stft, 
    calculate_match, evaluate_model, 
)

# We need to use an asd for whitening that does not include the glitch we want to reconstruct. 
# Therefore, we make a slight adjustment to PyCBC's whitening filter so that we can input our own psd
# You will see below that we calculate it using 14s of data adjacent to the 14s data segment that we use when preprocessing the glitch. 
# This ensures that the glitch should not appear in the asd and stop it being suppressed during whitening.
TimeSeries_pycbc.custom_whiten = custom_whiten


In [ ]:
model_name = 'DeepExtractor_257'
# Overall Evalutation GSPY
gspy_o3a_path = '/home/tom.dooney/data_o3a_high_confidence.csv'
gspy_o3b_path = '/home/tom.dooney/data_o3b_high_confidence.csv'

data_o3a = pd.read_csv(gspy_o3a_path)
data_o3b = pd.read_csv(gspy_o3b_path)

glitch_labels = list(data_o3a.label.unique())
# glitch_labels.remove('Chirp')
# glitch_labels.remove('Blip')
glitch_labels.remove('None_of_the_Above')

data_o3a = data_o3a.drop_duplicates()
data_o3b = data_o3b.drop_duplicates()

with open(DATA_PATH+'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
    
checkpoint_fine_tuned_f = CHECKPOINT_DIR+model_name+'/checkpoint_best_real_noise_base.pth.tar'
checkpoint_simulated_f = CHECKPOINT_DIR+model_name+'/checkpoint_best.pth.tar'

model_fine_tuned = UNET2D(in_channels=2, out_channels=2)
model_simulated = UNET2D(in_channels=2, out_channels=2)

# Move the model to the appropriate device (CPU or GPU)
model_fine_tuned.to(DEVICE)
model_simulated.to(DEVICE)

# If you have a checkpoint, load it
# if os.path.isfile(checkpoint_f):
checkpoint_fine_tuned = torch.load(checkpoint_fine_tuned_f, map_location=DEVICE)
checkpoint_simulated = torch.load(checkpoint_simulated_f, map_location=DEVICE)

model_fine_tuned.load_state_dict(checkpoint_fine_tuned['state_dict'])
model_simulated.load_state_dict(checkpoint_simulated['state_dict'])

# Ensure the model is in evaluation mode
model_fine_tuned.eval()
model_simulated.eval()

n_fft = 256*2  # FFT window size
win_length = n_fft//8  # Window length, same as n_fft for full window
hop_length = win_length // 2  # Hop length for overlap, commonly n_fft // 4
window = torch.hann_window(win_length)  # Hann window for smoother STFT

# datasets = [data_o3a, data_o3b]
# ifos = ['H1', 'L1']
# runs = ['O3a', 'O3b']

ifo = 'L1'
dataset = data_o3b
dataset_name = 'O3b'
glitch_type = 'Blip'

print(f'\nProcessing: Dataset={dataset_name}, IFO={ifo}, Glitch Type={glitch_type}\n')
data_filtered = dataset[dataset.ifo == ifo]
data_glitch = data_filtered[data_filtered.label == glitch_type]
data_glitch = data_glitch.drop_duplicates()

data_glitch.reset_index(drop=True, inplace=True)
num_samples = min(1, len(data_glitch))
# random_index = np.random.choice(len(data_glitch), size=1, replace=False)
random_index = 0

gps_time = data_glitch.GPStime.iloc[random_index]

try:
    segment_for_psd = TimeSeries.fetch_open_data(
        ifo,
        gps_time - 7-14,
        gps_time - 7,
        sample_rate=SAMPLE_RATE
    )
    
    glitch = TimeSeries.fetch_open_data(
        ifo,
        gps_time - 7,
        gps_time + 7,
        sample_rate=SAMPLE_RATE
    )

except Exception as e:
    print(f"Skipping GPS time {gps_time} for IFO {ifo} due to error: {e}")

segment_for_psd = np.asarray(segment_for_psd)
glitch = np.asarray(glitch)

segment_for_psd_pycbc = TimeSeries_pycbc(segment_for_psd, delta_t=1. / SAMPLE_RATE)
glitch_pycbc = TimeSeries_pycbc(glitch, delta_t=1. / SAMPLE_RATE)

white_segment_for_psd, psd = segment_for_psd_pycbc.whiten(2,1, remove_corrupted=False, return_psd=True)
white_glitch, _ = glitch_pycbc.custom_whiten(psd, return_psd=True)

if np.any(np.isnan(np.array(white_glitch))):
    print(f"Skipping GPS time {gps_time} for IFO {ifo} due to NaN values in the whitened data.")

T = len(white_glitch) / SAMPLE_RATE
t_inj = T / 2

glt_on_bkg_init = TimeSeries(white_glitch,
                             t0=-T/2,
                             sample_rate=SAMPLE_RATE,
                             name=ifo)

white_glitch = np.asarray(white_glitch)
white_glitch_centered = white_glitch[len(white_glitch) // 2 - 4096:len(white_glitch) // 2 + 4096]
white_glitch_centered_scaled = scaler.transform(white_glitch_centered.reshape(-1, 1)).reshape(white_glitch_centered.shape)

# Convert to PyTorch tensor
glitch_tensor = torch.tensor(white_glitch_centered_scaled, dtype=torch.float32)
# Convert to PyTorch tensor

# Apply STFT
stft_result = torch.stft(
    glitch_tensor, 
    n_fft=n_fft, 
    hop_length=hop_length, 
    win_length=win_length, 
    window=window, 
    return_complex=True
)

# Extract magnitude and phase
magnitude = torch.abs(stft_result)
phase = torch.angle(stft_result)

# Stack magnitude and phase into a 2-channel tensor
stft_mag_phase = torch.stack([magnitude, phase], dim=0)
stft_mag_phase = stft_mag_phase.unsqueeze(0)

# Convert scaled data to PyTorch tensors
h_stft = stft_mag_phase.float().to(DEVICE)

# Model prediction
with torch.no_grad():
    n_hat_stft_fine_tuned = model_fine_tuned(h_stft)
    n_hat_stft_simulated = model_simulated(h_stft)

n_hat_stft_fine_tuned = n_hat_stft_fine_tuned.cpu()
n_hat_stft_simulated = n_hat_stft_simulated.cpu()

# Separate magnitude and phase
magnitude_fine_tuned = n_hat_stft_fine_tuned[:, 0, :, :]  # First channel is the magnitude
phase_fine_tuned = n_hat_stft_fine_tuned[:, 1, :, :]      # Second channel is the phase
magnitude_simulated = n_hat_stft_simulated[:, 0, :, :]  # First channel is the magnitude
phase_simulated = n_hat_stft_simulated[:, 1, :, :]      # Second channel is the phase


# FROM HERE I NEED TO CHANGE TO THE TRANSFORMATIONS FOR THE THREE MODEL OUTPUTS
# 1. Model Output: UNET (no_glitch)
real_part_fine_tuned = magnitude_fine_tuned * torch.cos(phase_fine_tuned)
imag_part_fine_tuned = magnitude_fine_tuned * torch.sin(phase_fine_tuned)
stft_complex_fine_tuned = torch.complex(real_part_fine_tuned, imag_part_fine_tuned)

# Perform the ISTFT (inverse STFT) for UNET (no_glitch)
n_hat_t_scaled_fine_tuned = torch.istft(
    stft_complex_fine_tuned,
    n_fft=n_fft,
    hop_length=hop_length,
    win_length=win_length,
    window=window,
)
n_hat_t_scaled_fine_tuned = n_hat_t_scaled_fine_tuned.numpy().squeeze()

# Scale back the inverse transformed data
n_hat_t_fine_tuned = scaler.inverse_transform(
    n_hat_t_scaled_fine_tuned.reshape(-1, n_hat_t_scaled_fine_tuned.shape[-1])
).reshape(n_hat_t_scaled_fine_tuned.shape)

# 2. Model Output: Simulated
real_part_simulated = magnitude_simulated * torch.cos(phase_simulated)
imag_part_simulated = magnitude_simulated * torch.sin(phase_simulated)
stft_complex_simulated = torch.complex(real_part_simulated, imag_part_simulated)

n_hat_t_scaled_simulated = torch.istft(
    stft_complex_simulated,
    n_fft=n_fft,
    hop_length=hop_length,
    win_length=win_length,
    window=window,
)
n_hat_t_scaled_simulated = n_hat_t_scaled_simulated.numpy().squeeze()

# Scale back the inverse transformed data
n_hat_t_simulated = scaler.inverse_transform(
    n_hat_t_scaled_simulated.reshape(-1, n_hat_t_scaled_simulated.shape[-1])
).reshape(n_hat_t_scaled_simulated.shape)

 # Calculate differences
g_hat_fine_tuned = white_glitch_centered - n_hat_t_fine_tuned
g_hat_simulated = white_glitch_centered - n_hat_t_simulated

# Subtraction of Difference_specs_val from respective white_glitch data
white_glitch_subtract_fine_tuned = white_glitch.copy()
white_glitch_subtract_fine_tuned[len(white_glitch_subtract_fine_tuned) // 2 - 4096:len(white_glitch_subtract_fine_tuned) // 2 + 4096] -= g_hat_fine_tuned

white_glitch_subtract_simulated = white_glitch.copy()
white_glitch_subtract_simulated[len(white_glitch_subtract_simulated) // 2 - 4096:len(white_glitch_subtract_simulated) // 2 + 4096] -= g_hat_simulated

# Titles and data for the four subplots
figure_titles = [
    'Input Q-Scan',
    'Mitigation without Transfer Learning',
    'Mitigation with Transfer Learning',
    'Time-Domain Reconstructions'
]

data_to_plot = [
    (white_glitch, t_inj, 'Input'),
    (white_glitch_subtract_simulated, t_inj, 'Simulated'),
    (white_glitch_subtract_fine_tuned, t_inj, 'Fine-tuned'),
    (white_glitch_centered, g_hat_simulated, g_hat_fine_tuned)
]

# Create a single figure with 4 subplots in a row
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

t = np.linspace(0, 2, 8192)  # Time axis for plots

for ax, title, data in zip(axes, figure_titles, data_to_plot):
    if title == 'Time-Domain Reconstructions':
        ax.plot(t, data[0], label='Input', c='gray', alpha=0.4)
        ax.plot(t, data[1], label='Without Transfer Learning', c='C0', linestyle=':', linewidth=2, alpha=0.7)
        ax.plot(t, data[2], label='With Transfer Learning', c='C2', linestyle=':', linewidth=2, alpha=0.8)
        ax.set_xlabel('Time (s)', fontsize=20)
        ax.set_ylabel('Amplitude', fontsize=20)
        ax.legend(fontsize=16)
        ax.grid(False)

    elif title == 'Mitigation without Transfer Learning':
        plot_q_transform(data[0], crop=(data[1], 2), srate=SAMPLE_RATE, whiten=False, ax=ax)
        ax.set_xlabel('Time [s]', fontsize=20)
        ax.set_ylabel('Frequency [Hz]', fontsize=20)
        
        # Add red box highlight
        lower_bound = np.sqrt(256 * 512)
        upper_bound = np.sqrt(512 * 1024)
        ax.add_patch(plt.Rectangle(
            (0, lower_bound),  # Bottom-left corner (x, y)
            2,  # Width (entire x-axis range)
            upper_bound - lower_bound,  # Height
            linewidth=2, edgecolor='red', facecolor='none'
        ))

    else:
        plot_q_transform(data[0], crop=(data[1], 2), srate=SAMPLE_RATE, whiten=False, ax=ax)
        ax.set_xlabel('Time [s]', fontsize=20)
        ax.set_ylabel('Frequency [Hz]', fontsize=20)

    ax.set_title(title, fontsize=16)

# Adjust layout for better spacing
plt.tight_layout()
plt.show()
